# <span style="color:blue">PRÉ PROCESSAMENTO DE DADOS</span> #

## <span style="color:blue">PACOTES UTILIZADOS</span> ##

In [2]:
import pandas as pd
import numpy as np
from IPython.display import display

## <span style="color:blue">OBTENÇÃO DE DADOS BRUTOS E TRANSFORMAÇÃO EM DATASET PRIMÁRIO</span> ## 

In [5]:
# ============================================================
# 1. CARREGAR AS DUAS BASES ORIGINAIS
# ============================================================

train = pd.read_csv("/projeto_tcc_2026/dados/dados_primarios/fraudTrain.csv")
test = pd.read_csv("/projeto_tcc_2026/dados/dados_primarios/fraudTest.csv")

print("DIMENSÕES DAS BASES ORIGINAIS")
print("=" * 100)

print("fraudTrain:", train.shape)
print("fraudTest: ", test.shape)


# ============================================================
# 2. VERIFICAR SE AS FEATURES SÃO IGUAIS
# ============================================================

if list(train.columns) == list(test.columns):
    print("\nAs duas bases possuem exatamente as mesmas features.")
else:
    print("\nATENÇÃO: As bases possuem diferenças nas features.")

    print("\nFeatures somente no Train:")
    print(set(train.columns) - set(test.columns))

    print("\nFeatures somente no Test:")
    print(set(test.columns) - set(train.columns))


# ============================================================
# 3. REMOVER O IDENTIFICADOR ANTIGO
# ============================================================

if "Unnamed: 0" in train.columns:
    train = train.drop(columns=["Unnamed: 0"])

if "Unnamed: 0" in test.columns:
    test = test.drop(columns=["Unnamed: 0"])


# ============================================================
# 4. JUNTAR TRAIN E TEST
# ============================================================

dataset_primario = pd.concat(
    [train, test],
    axis=0,
    ignore_index=True
)


# ============================================================
# 5. CRIAR NOVO IDENTIFICADOR ÚNICO
# ============================================================

dataset_primario.insert(
    0,
    "NID",
    range(1, len(dataset_primario) + 1)
)


# ============================================================
# 6. VERIFICAR O NOVO IDENTIFICADOR
# ============================================================

print("\nVERIFICAÇÃO DO NID")
print("=" * 100)

print("Primeiro NID:", dataset_primario["NID"].min())
print("Último NID:", dataset_primario["NID"].max())
print("Quantidade de NIDs únicos:", dataset_primario["NID"].nunique())
print("Quantidade de NIDs duplicados:", dataset_primario["NID"].duplicated().sum())


# ============================================================
# 7. DIMENSÃO FINAL
# ============================================================

print("\nDIMENSÃO DO DATASET PRIMÁRIO")
print("=" * 100)

print("Linhas:", dataset_primario.shape[0])
print("Features:", dataset_primario.shape[1])


# ============================================================
# 8. SALVAR O DATASET PRIMÁRIO
# ============================================================

dataset_primario.to_csv(
    "/projeto_tcc_2026/dados/dados_primarios/dataset_primario.csv",
    index=False
)

print("\nArquivo 'dataset_primario.csv' criado com sucesso!")

DIMENSÕES DAS BASES ORIGINAIS
fraudTrain: (1296675, 23)
fraudTest:  (555719, 23)

As duas bases possuem exatamente as mesmas features.

VERIFICAÇÃO DO NID
Primeiro NID: 1
Último NID: 1852394
Quantidade de NIDs únicos: 1852394
Quantidade de NIDs duplicados: 0

DIMENSÃO DO DATASET PRIMÁRIO
Linhas: 1852394
Features: 23

Arquivo 'dataset_primario.csv' criado com sucesso!


## <span style="color:blue">TRANSFORMAÇÃO DO DATASET PRIMÁRIO EM DATASET FINAL</span> ## 

In [7]:
# ============================================================
# 1. CARREGAR O DATASET PRIMÁRIO
# ============================================================

df = pd.read_csv("/projeto_tcc_2026/dados/dados_primarios/dataset_primario.csv")

print("Dimensão inicial:", df.shape)


# ============================================================
# 2. VERIFICAR O IDENTIFICADOR NID
# ============================================================

print("\nVERIFICAÇÃO INICIAL DO NID")
print("=" * 100)

print("Primeiro NID:", df["NID"].min())
print("Último NID:", df["NID"].max())
print("NIDs únicos:", df["NID"].nunique())
print("NIDs duplicados:", df["NID"].duplicated().sum())


# ============================================================
# 3. REMOVER FEATURES NÃO UTILIZADAS
# ============================================================

colunas_remover = [
    "city",
    "state",
    "zip",
    "trans_num",
    "unix_time",
    "street"
]

df = df.drop(
    columns=[
        coluna
        for coluna in colunas_remover
        if coluna in df.columns
    ]
)


# ============================================================
# 4. RENOMEAR TARGET
# ============================================================

if "is_fraud" in df.columns:
    df = df.rename(
        columns={"is_fraud": "TARGET"}
    )

elif "is_fraude" in df.columns:
    df = df.rename(
        columns={"is_fraude": "TARGET"}
    )


# ============================================================
# 5. TRATAR DATA E HORA DA TRANSAÇÃO
# ============================================================

df["trans_date_trans_time"] = pd.to_datetime(
    df["trans_date_trans_time"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)


# ------------------------------------------------------------
# DIA DO MÊS
# ------------------------------------------------------------

df["TRANS_DAY"] = (
    df["trans_date_trans_time"]
    .dt.day
)


# ------------------------------------------------------------
# DIA DA SEMANA
# ------------------------------------------------------------

mapa_dias = {
    0: "Segunda",
    1: "Terca",
    2: "Quarta",
    3: "Quinta",
    4: "Sexta",
    5: "Sabado",
    6: "Domingo"
}

df["TRANS_WEEK"] = (
    df["trans_date_trans_time"]
    .dt.dayofweek
    .map(mapa_dias)
)


# ------------------------------------------------------------
# ANO
# ------------------------------------------------------------

df["TRANS_YEAR"] = (
    df["trans_date_trans_time"]
    .dt.year
)


# ------------------------------------------------------------
# MÊS EM SENO E COSSENO
# ------------------------------------------------------------

mes = df["trans_date_trans_time"].dt.month

df["TRANS_MONTH_SEN"] = np.sin(
    2 * np.pi * (mes - 1) / 12
)

df["TRANS_MONTH_COS"] = np.cos(
    2 * np.pi * (mes - 1) / 12
)


# ------------------------------------------------------------
# HORÁRIO EM SENO E COSSENO
# Considerando hora + minuto + segundo
# ------------------------------------------------------------

hora_decimal = (
    df["trans_date_trans_time"].dt.hour
    + df["trans_date_trans_time"].dt.minute / 60
    + df["trans_date_trans_time"].dt.second / 3600
)

df["TRANS_HOUR_SEN"] = np.sin(
    2 * np.pi * hora_decimal / 24
)

df["TRANS_HOUR_COS"] = np.cos(
    2 * np.pi * hora_decimal / 24
)


# ------------------------------------------------------------
# REMOVER DATA/HORA ORIGINAL
# ------------------------------------------------------------

df = df.drop(
    columns=["trans_date_trans_time"]
)


# ============================================================
# 6. RENOMEAR FEATURES DA TRANSAÇÃO E DO REMETENTE
# ============================================================

df = df.rename(
    columns={
        "amt": "TRANS_VALUE",
        "cc_num": "TRANS_NUM_CARD",

        "job": "SEND_JOB",
        "gender": "SEND_GENDER",

        "lat": "SEND_LAT_REGISTER",
        "long": "SEND_LONG_REGISTER",
        "city_pop": "SEND_POP_REGISTER"
    }
)


# ============================================================
# 7. CRIAR NOME COMPLETO DO REMETENTE
# ============================================================

df["SEND_NAME"] = (
    df["first"]
    .fillna("")
    .astype(str)
    .str.strip()

    + " "

    + df["last"]
    .fillna("")
    .astype(str)
    .str.strip()
).str.strip()


# ============================================================
# 8. TRANSFORMAR DATA DE NASCIMENTO EM IDADE
# ============================================================

df["dob"] = pd.to_datetime(
    df["dob"],
    errors="coerce"
)


# ------------------------------------------------------------
# DATA DE REFERÊNCIA FIXA
# ------------------------------------------------------------

data_referencia = pd.Timestamp("2026-09-03")


# ------------------------------------------------------------
# IDADE EM ANOS DECIMAIS
# ------------------------------------------------------------

df["SEND_AGE"] = (
    (data_referencia - df["dob"]).dt.total_seconds()
    / (365.2425 * 24 * 60 * 60)
).round(2)


# ------------------------------------------------------------
# REMOVER FEATURES ORIGINAIS
# ------------------------------------------------------------

df = df.drop(
    columns=[
        "first",
        "last",
        "dob"
    ]
)


# ============================================================
# 9. RENOMEAR FEATURES DO RECEBEDOR
# ============================================================

df = df.rename(
    columns={
        "merchant": "RECIVE_LOC",
        "category": "RECIVE_CATEGORY",
        "merch_lat": "RECIVE_LAT",
        "merch_long": "RECIVE_LONG"
    }
)


# ============================================================
# 10. ADICIONAR NOMENCLATURA DOS FUTUROS ENCODERS
# ============================================================

df = df.rename(
    columns={

        # ----------------------------------------------------
        # IDENTIFICADOR
        # ----------------------------------------------------

        "NID": "NID_ALPHA",

        # ----------------------------------------------------
        # FUTURO FREQUENCY ENCODING
        # ----------------------------------------------------

        "TRANS_NUM_CARD": "TRANS_NUM_CARD_FEWF",
        "RECIVE_LOC": "RECIVE_LOC_FEWF",
        "SEND_JOB": "SEND_JOB_FEWF",
        "SEND_NAME": "SEND_NAME_FEWF",

        # ----------------------------------------------------
        # FUTURO BINARY ENCODING
        # ----------------------------------------------------

        "SEND_GENDER": "SEND_GENDER_BE",
        "TRANS_YEAR": "TRANS_YEAR_BE",

        # ----------------------------------------------------
        # FUTURO ONE-HOT ENCODING
        # COM TRATAMENTO DE CATEGORIAS DESCONHECIDAS
        # ----------------------------------------------------

        "RECIVE_CATEGORY": "RECIVE_CATEGORY_OHEWI",
        "TRANS_WEEK": "TRANS_WEEK_OHEWI",

        # ----------------------------------------------------
        # VARIÁVEL RESPOSTA
        # ----------------------------------------------------

        "TARGET": "TARGET_OMEGA"
    }
)


# ============================================================
# 11. ORGANIZAR NID PRIMEIRO E TARGET POR ÚLTIMO
# ============================================================

colunas_meio = [
    coluna
    for coluna in df.columns
    if coluna not in [
        "NID_ALPHA",
        "TARGET_OMEGA"
    ]
]

df = df[
    ["NID_ALPHA"]
    + colunas_meio
    + ["TARGET_OMEGA"]
]


# ============================================================
# 12. VERIFICAÇÕES DO DATASET FINAL
# ============================================================

print("\n" + "=" * 100)
print("DATASET FINAL")
print("=" * 100)

print(f"Linhas: {df.shape[0]}")
print(f"Features: {df.shape[1]}")


# ============================================================
# 13. MOSTRAR FEATURES FINAIS
# ============================================================

print("\nFEATURES FINAIS")
print("=" * 100)

for i, coluna in enumerate(
    df.columns,
    start=1
):
    print(f"{i}. {coluna}")


# ============================================================
# 14. VERIFICAR SEND_AGE
# ============================================================

print("\nVERIFICAÇÃO DE SEND_AGE")
print("=" * 100)

print(
    "Data de referência utilizada:",
    data_referencia.strftime("%d/%m/%Y")
)

print(
    "Menor idade:",
    df["SEND_AGE"].min()
)

print(
    "Maior idade:",
    df["SEND_AGE"].max()
)


# ============================================================
# 15. VERIFICAR TARGET
# ============================================================

print("\nVERIFICAÇÃO DO TARGET")
print("=" * 100)

print(
    df["TARGET_OMEGA"]
    .value_counts()
    .sort_index()
)


# ============================================================
# 16. SALVAR DATASET FINAL
# ============================================================


df.to_csv(
    "/projeto_tcc_2026/dados/dataset_final/dataset_final.csv",
    index=False
)

Dimensão inicial: (1852394, 23)

VERIFICAÇÃO INICIAL DO NID
Primeiro NID: 1
Último NID: 1852394
NIDs únicos: 1852394
NIDs duplicados: 0

DATASET FINAL
Linhas: 1852394
Features: 22

FEATURES FINAIS
1. NID_ALPHA
2. TRANS_NUM_CARD_FEWF
3. RECIVE_LOC_FEWF
4. RECIVE_CATEGORY_OHEWI
5. TRANS_VALUE
6. SEND_GENDER_BE
7. SEND_LAT_REGISTER
8. SEND_LONG_REGISTER
9. SEND_POP_REGISTER
10. SEND_JOB_FEWF
11. RECIVE_LAT
12. RECIVE_LONG
13. TRANS_DAY
14. TRANS_WEEK_OHEWI
15. TRANS_YEAR_BE
16. TRANS_MONTH_SEN
17. TRANS_MONTH_COS
18. TRANS_HOUR_SEN
19. TRANS_HOUR_COS
20. SEND_NAME_FEWF
21. SEND_AGE
22. TARGET_OMEGA

VERIFICAÇÃO DE SEND_AGE
Data de referência utilizada: 03/09/2026
Menor idade: 21.59
Maior idade: 101.84

VERIFICAÇÃO DO TARGET
TARGET_OMEGA
0    1842743
1       9651
Name: count, dtype: int64
